In [ ]:
import warnings
from pathlib import Path

import prism

from imagematerials.concepts import create_building_graph, create_vehicle_graph
from imagematerials.factory import ModelFactory
from imagematerials.model import GenericMaterials, GenericStocks, MaterialIntensities
from imagematerials.util import export_to_netcdf, import_from_netcdf, rebroadcast_prep_data
from imagematerials.vehicles import (
    preprocess,
)
import numpy as np

In [2]:
def get_vema_prep():
    base_dir = "../data/raw"
    prep_fp = Path("prep_vema.nc")
    if not prep_fp.is_file():
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            orig_prep_data = preprocess(base_dir)
        export_to_netcdf(orig_prep_data, prep_fp)

    prep_data = import_from_netcdf(prep_fp)
    share_coords = set()
    for cur_type in prep_data["shares"].Type.values:
        share_coords.add(cur_type.split(" - ")[0])
    output_coords_type = [x for x in prep_data["stocks"].Type.values if x not in share_coords] + list(prep_data["shares"].coords["Type"].values)
    prep_data.pop("shares")
    knowledge_graph = create_vehicle_graph()
    new_prep_data = rebroadcast_prep_data(prep_data, knowledge_graph, dim="Type", output_coords=output_coords_type)
    region_coords = np.sort(prep_data["stocks"].coords["Region"].values.astype(int)).astype(str)[:-2]
    new_prep_data = rebroadcast_prep_data(new_prep_data, knowledge_graph, dim="Region", output_coords=region_coords)
    new_prep_data["knowledge_graph"] = knowledge_graph

    new_prep_data["weights"] = new_prep_data.pop("vehicle_weights")
    return new_prep_data

In [3]:
def get_buildings_prep():
    base_dir = Path("..", "IMAGE-Mat_old_version", "IMAGE-Mat", "BUMA")
    prep_fp = Path("prep_buildings.nc")
    if not prep_fp.is_file():
        prep_data = preprocess(base_dir)
        export_to_netcdf(prep_data, prep_fp)
    else:
        prep_data = import_from_netcdf(prep_fp)
    new_prep_data = {k: v for k, v in prep_data.items()}
    new_prep_data["knowledge_graph"] = create_building_graph()
    return new_prep_data

In [4]:
vhc_prep = get_vema_prep()
bld_prep = get_buildings_prep()

In [5]:
# Define the complete timeline, including historic tail
# time_start = prep_data["stocks"].coords["Time"].min().values
time_start = 1960
complete_timeline = prism.Timeline(time_start, 2060, 1)
simulation_timeline = prism.Timeline(1970, 2060, 1)

In [6]:
REGION = prism.Dimension("Region")
MATERIAL_TYPE = prism.Dimension("material")

@prism.interface
class SumMaterials(prism.Model):
    Region: prism.Coords[REGION]
    material: prism.Coords[MATERIAL_TYPE]

    input_data: tuple[str] = ("inflow_materials", )
    output_data: tuple[str] = ("summed_inflow_materials", )



    summed_inflow_materials: prism.TimeVariable[REGION, MATERIAL_TYPE, "count"] = prism.export()

    def compute_initial_values(self, time: prism.Timeline):
        pass

    def compute_values(self, time: prism.Time, inflow_materials):
        t, dt = time.t, time.dt
        self.summed_inflow_materials[t].loc[{"material": ["Wood", "Steel"]}] = 0
        for inflow in inflow_materials:
            self.summed_inflow_materials[t].loc[{"material": ["Wood", "Steel"]}] += inflow[t].sel(material=["Wood", "Steel"]).sum("Type")

In [7]:
factory = ModelFactory(
    {"bld": bld_prep, "vhc": vhc_prep, "combi": {}}, complete_timeline
    ).add(GenericStocks, ["bld", "vhc"]
    ).add(GenericMaterials,  "vhc"
    ).add(MaterialIntensities, "bld",
    ).add(SumMaterials, "combi", input_sources={"inflow_materials": ["vhc", "bld"]}
    )
model = factory.finish()

In [10]:
import warnings
with warnings.catch_warnings():
    warnings.filterwarnings("ignore")
    model.simulate(simulation_timeline)

In [14]:
model.combi["summed_inflow_materials"][2020].sel(material=["Wood", "Steel"])

Magnitude,[[118463.63771889317 1752754796.525083] [2120169.7503986303 55073659958.25919] [57815.96975603416 2125298646.9076915] [143820.0574115615 4850359388.408313] [247142.2164763471 6693850210.345263] [188937.68354699342 9980358367.541069] [4705377.62254448 4346341837.343484] [8137793.1194333555 6384450488.606569] [12394457.514133321 5866622905.444465] [68648.92318774718 2068642217.4246812] [110656.48675022977 8180323299.369704] [556.7675540515728 1057816071.7801281] [537367.1367678401 2911865719.9579124] [1849.5704892416059 435865592.67394066] [166378.38484991345 1756251912.1672542] [921.3881197882926 230273190.97209728] [3348.950708302803 15938949599.78044] [5908.419954920257 9228591309.787739] [18839597.08247192 6751404262.01005] [4126709.286889879 67637957987.76407] [584198.5967073713 10764548165.976524] [742642.2088491358 10207208173.73009] [31098744.158313036 5799062985.7080145] [17135.073556914773 1392875916.3045712] [62531.561889776734 6947513124.941193] [4315805.70532656 3306016779.7338686]]
Units,count
